# Build the evidence knowledge graph

Run each protected stage deliberately. Every write stage below documents its inputs, outputs, database effect, rerun behavior, and invalidation rules; the following cell is read-only inspection.

In [ ]:
from pathlib import Path
from potency_kg.alias_discovery import DEFAULT_ENABLE_LLM_ALIAS_DISCOVERY
from potency_kg.workflow_stages import (
    classify_jev_stage, discover_aliases_deterministic_stage, discover_aliases_llm_stage,
    embed_stage, export_stage, extract_stage, import_stage, inspect_aliases_stage,
    inspect_alias_evidence_stage, inspect_stage, load_alias_seeds_stage, materialize_stage,
    preflight_stage, refresh_graph_stage
)
working_directory = Path.cwd().resolve()
package_root = working_directory.parent if working_directory.name == 'notebooks' else working_directory
config_path = package_root / 'configs/casgevy-potency.json'
output_root = package_root / 'results/kg'
enable_llm_alias_discovery = DEFAULT_ENABLE_LLM_ALIAS_DISCOVERY


## Preflight
Inputs: supplied descriptor and corpus. Outputs: validated hashes and counts. Database effect: none. Rerun behavior: safe and read-only. Invalidation: changed supplied inputs require a new import.

In [ ]:
preflight_report = preflight_stage(config_path)
preflight_report


In [ ]:
preflight_inspection = {'document_count': getattr(preflight_report, 'document_count', None)}
preflight_inspection

## Structural import
Inputs: preflight-valid descriptor. Outputs: source and passage report. Database effect: writes structural rows. Rerun behavior: idempotent. Invalidation: a new source version requires later alias and semantic stages.

In [ ]:
import_report = import_stage(config_path)
import_report


In [ ]:
import_inspection = inspect_stage(config_path, output_root=output_root)
import_inspection


## Static aliases
Inputs: authoritative seed file. Outputs: seed reconciliation report. Database effect: writes approved static aliases and evidence. Rerun behavior: idempotent. Invalidation: catalog changes require later semantic reruns.

In [ ]:
seed_report = load_alias_seeds_stage(config_path)
seed_report


In [ ]:
seed_inspection = inspect_stage(config_path, output_root=output_root)
seed_inspection


## Deterministic discovery
Inputs: active passages. Outputs: pending alias proposals and evidence. Database effect: writes deterministic runs. Rerun behavior: reuses completed runs. Invalidation: new source versions need discovery.

In [ ]:
deterministic_report = discover_aliases_deterministic_stage(config_path)
deterministic_report


In [ ]:
deterministic_inspection = inspect_stage(config_path, output_root=output_root)
deterministic_inspection

See aliases data in http://localhost:8888/lab/tree/notebooks/03_review_aliases.ipynb

## Optional LLM discovery
Inputs: active passages and an explicitly enabled model gate. Outputs: pending proposals. Database effect: writes audited LLM discovery runs. Rerun behavior: reuses matching runs. Invalidation: catalog decisions require semantic reruns.

In [ ]:
llm_discovery_report = discover_aliases_llm_stage(config_path, enabled=enable_llm_alias_discovery)
llm_discovery_report


In [ ]:
llm_discovery_inspection = inspect_stage(config_path, output_root=output_root)
llm_discovery_inspection


## Alias inspection and review
Inputs: pending aliases and evidence. Outputs: read-only review queue. Database effect: none here; review is explicit. Rerun behavior: safe. Invalidation: approval requires an explicit later semantic rebuild.

Use `python -m potency_kg.cli review-alias --config configs/casgevy-potency.json --alias-id ID --decision approved --reviewer NAME --entity-type Product --canonical-key KEY --canonical-name NAME` after inspecting evidence.

In [ ]:
review_queue = inspect_aliases_stage(config_path, status='pending')
selected_alias_id = review_queue[0].alias_id if review_queue else None
review_evidence = inspect_alias_evidence_stage(config_path, alias_id=selected_alias_id) if selected_alias_id else ()
{'aliases': review_queue, 'selected_alias_id': selected_alias_id, 'evidence': review_evidence}


In [ ]:
review_inspection = inspect_stage(config_path, output_root=output_root)
review_inspection


## Embeddings
Inputs: active passages and explicit embedding settings. Outputs: embedding report. Database effect: writes vectors. Rerun behavior: reuses matching runs. Invalidation: source changes require embeddings again.

In [ ]:
embedding_report = embed_stage(config_path)
embedding_report


In [ ]:
embedding_inspection = inspect_stage(config_path, output_root=output_root)
embedding_inspection


## Jev
Inputs: embeddings and explicit Jev settings. Outputs: selection report. Database effect: writes classifications. Rerun behavior: reuses matching responses. Invalidation: alias catalog changes require Jev again.

In [ ]:
jev_report = classify_jev_stage(config_path)
jev_report


In [ ]:
jev_inspection = inspect_stage(config_path, output_root=output_root)
jev_inspection


## Extraction
Inputs: selected passages and explicit generation settings. Outputs: structured extraction report. Database effect: writes validated runs. Rerun behavior: reuses exact runs. Invalidation: alias catalog changes require extraction again.

In [ ]:
extraction_report = extract_stage(config_path)
extraction_report


In [ ]:
extraction_inspection = inspect_stage(config_path, output_root=output_root)
extraction_inspection


## Materialization
Inputs: validated extraction runs. Outputs: materialization report. Database effect: writes relational graph facts. Rerun behavior: idempotent. Invalidation: new semantic runs require materialization.

In [ ]:
materialization_report = materialize_stage(config_path)
materialization_report


In [ ]:
materialization_inspection = inspect_stage(config_path, output_root=output_root)
materialization_inspection


## Refresh
Inputs: materialized relational state. Outputs: activated graph report. Database effect: writes a shadow graph. Rerun behavior: creates or reuses deterministic state. Invalidation: semantic inputs require refresh.

In [ ]:
refresh_report = refresh_graph_stage(config_path, mode='semantic')
refresh_report


In [ ]:
refresh_inspection = inspect_stage(config_path, output_root=output_root)
refresh_inspection


## Export
Inputs: active graph. Outputs: immutable export report. Database effect: records publication state. Rerun behavior: validates and reuses an identical export. Invalidation: a new graph needs export.

In [ ]:
export_report = export_stage(config_path, output_root)
export_report


In [ ]:
export_inspection = inspect_stage(config_path, output_root=output_root)
export_inspection


## Inspection
Inputs: current scope. Outputs: catalog status. Database effect: none. Rerun behavior: safe and read-only. Invalidation: reports current catalog and semantic staleness.

In [ ]:
inspection_report = inspect_stage(config_path, output_root=output_root)
inspection_report
